# Exploration of GPT-2 small and mechinterp methods
## Transformerlens introduction

In the following notebook I try to get familiar with using transformerlens to analyse GPT-2 structure and activations.

In [1]:
import sys, torch
from importlib.metadata import version
print(sys.executable)
print("torch", torch.__version__, "| mps:", torch.backends.mps.is_available())
print("transformer_lens", version("transformer_lens"))

/Users/Brose/dev/private_repos/mechinterp_xai/.venv/bin/python
torch 2.13.0 | mps: True
transformer_lens 3.5.1


In [2]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2", device="mps")

/Users/Brose/dev/private_repos/mechinterp_xai/.venv/lib/python3.13/site-packages/transformer_lens/config/hooked_transformer_config.py:354: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.13.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer


Im using GPT2-small since its small enough to run it fast locally and it might be the best supported model in Transformerlens.

In [3]:
model.cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'NTK_original_ctx_len': 8192,
 'act_fn': 'gelu_new',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(8.0),
 'attn_scores_soft_cap': -1.0,
 'attn_types': None,
 'checkpoint_index': None,
 'checkpoint_label_type': None,
 'checkpoint_value': None,
 'd_head': 64,
 'd_mlp': 3072,
 'd_model': 768,
 'd_vocab': 50257,
 'd_vocab_out': 50257,
 'decoder_start_token_id': None,
 'default_prepend_bos': True,
 'device': 'mps',
 'dtype': torch.float32,
 'eps': 1e-05,
 'experts_per_token': None,
 'final_rms': False,
 'from_checkpoint': False,
 'gated_mlp': False,
 'init_mode': 'gpt2',
 'init_weights': False,
 'initializer_range': np.float64(0.02886751345948129),
 'layer_norm_folding': False,
 'load_in_4bit': False,
 'model_name': 'gpt2',
 'n_ctx': 1024,
 'n_devices': 1,
 'n_heads': 12,
 'n_key_value_heads': None,
 'n_layers': 12,
 'n_params': 849

Inspect the dimensions of the tensors:

In [4]:
for name in ["W_E", "W_pos", "W_Q", "W_K", "W_V", "W_O", "W_in", "W_out", "W_U"]:
    print(f"{name:6s} {tuple(getattr(model, name).shape)}")

W_E    (50257, 768)
W_pos  (1024, 768)
W_Q    (12, 12, 768, 64)
W_K    (12, 12, 768, 64)
W_V    (12, 12, 768, 64)
W_O    (12, 12, 64, 768)
W_in   (12, 768, 3072)
W_out  (12, 3072, 768)
W_U    (768, 50257)


Closer look into the first layer:

In [19]:
model.blocks[0]

TransformerBlock(
  (ln1): LayerNormPre(
    (hook_scale): HookPoint(name='blocks.0.ln1.hook_scale')
    (hook_normalized): HookPoint(name='blocks.0.ln1.hook_normalized')
  )
  (ln2): LayerNormPre(
    (hook_scale): HookPoint(name='blocks.0.ln2.hook_scale')
    (hook_normalized): HookPoint(name='blocks.0.ln2.hook_normalized')
  )
  (attn): Attention(
    (hook_k): HookPoint(name='blocks.0.attn.hook_k')
    (hook_q): HookPoint(name='blocks.0.attn.hook_q')
    (hook_v): HookPoint(name='blocks.0.attn.hook_v')
    (hook_z): HookPoint(name='blocks.0.attn.hook_z')
    (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
    (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
    (hook_result): HookPoint(name='blocks.0.attn.hook_result')
  )
  (mlp): MLP(
    (hook_pre): HookPoint(name='blocks.0.mlp.hook_pre')
    (hook_post): HookPoint(name='blocks.0.mlp.hook_post')
  )
  (hook_attn_in): HookPoint(name='blocks.0.hook_attn_in')
  (hook_q_input): HookPoint(name='blocks

In [6]:
prompt = "The capital of France is Paris. Interpretability researchers"
tokens = model.to_tokens(prompt)
print(tokens.shape)
print(model.to_str_tokens(prompt))

torch.Size([1, 11])
['<|endoftext|>', 'The', ' capital', ' of', ' France', ' is', ' Paris', '.', ' Interpret', 'ability', ' researchers']


In [7]:
logits, cache = model.run_with_cache(tokens)
print(logits.shape)
for name, act in cache.items():
    if not name.startswith("blocks.") or name.startswith("blocks.0."):
        print(f"{name:40s} {tuple(act.shape)}")

torch.Size([1, 11, 50257])
hook_embed                               (1, 11, 768)
hook_pos_embed                           (1, 11, 768)
blocks.0.hook_resid_pre                  (1, 11, 768)
blocks.0.ln1.hook_scale                  (1, 11, 1)
blocks.0.ln1.hook_normalized             (1, 11, 768)
blocks.0.attn.hook_q                     (1, 11, 12, 64)
blocks.0.attn.hook_k                     (1, 11, 12, 64)
blocks.0.attn.hook_v                     (1, 11, 12, 64)
blocks.0.attn.hook_attn_scores           (1, 12, 11, 11)
blocks.0.attn.hook_pattern               (1, 12, 11, 11)
blocks.0.attn.hook_z                     (1, 11, 12, 64)
blocks.0.hook_attn_out                   (1, 11, 768)
blocks.0.hook_resid_mid                  (1, 11, 768)
blocks.0.ln2.hook_scale                  (1, 11, 1)
blocks.0.ln2.hook_normalized             (1, 11, 768)
blocks.0.mlp.hook_pre                    (1, 11, 3072)
blocks.0.mlp.hook_post                   (1, 11, 3072)
blocks.0.hook_mlp_out                  

In [8]:
import torch
c = cache
print(torch.allclose(c["hook_embed"] + c["hook_pos_embed"], c["blocks.0.hook_resid_pre"]))
print(torch.allclose(c["blocks.0.hook_resid_pre"] + c["blocks.0.hook_attn_out"], c["blocks.0.hook_resid_mid"]))
print(torch.allclose(c["blocks.0.hook_resid_mid"] + c["blocks.0.hook_mlp_out"], c["blocks.0.hook_resid_post"]))
print(torch.allclose(c["blocks.0.hook_resid_post"], c["blocks.1.hook_resid_pre"]))
print(c["blocks.0.attn.hook_pattern"][0, 0].sum(-1))
print(c["blocks.0.attn.hook_pattern"][0, 0].round(decimals=2))

True
True
True
True
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000], device='mps:0')
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.9300, 0.0700, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.7500, 0.1600, 0.0900, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.6900, 0.1600, 0.1100, 0.0400, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.5700, 0.1800, 0.1100, 0.0900, 0.0600, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.6600, 0.1600, 0.0700, 0.0300, 0.0500, 0.0400, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.4100, 0.1400, 0.0600, 0.0600, 0.1200, 0.0600, 0.1500, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.6300, 0.1500, 0.0600, 0.0100, 0.0500, 0.0200, 0.0700, 0.0100, 0.0000,
         0.0000, 0.0000],
   

In [9]:
from circuitsvis.attention import attention_heads

str_tokens = model.to_str_tokens(prompt)
attention_heads(
    attention=cache["blocks.0.attn.hook_pattern"][0].cpu(),  # [head, dest, src], Batch-Index weg
    tokens=str_tokens,
)

In [ ]:
str_tokens = model.to_str_tokens(prompt)
attention_heads(
    attention=cache["blocks.11.attn.hook_pattern"][0].cpu(),  # last layer
    tokens=str_tokens,
)

Logit lens: treat the residual stream before each layer as if it were the final one

In [ ]:
resid, labels = cache.accumulated_resid(apply_ln=True, return_labels=True)  # [13, 1, 11, 768]
lens_logits = resid[:, 0, -1, :] @ model.W_U  # last position, all vocab entries per layer
for lab, tok in zip(labels, lens_logits.argmax(-1)):
    print(f"{lab:12s} {model.to_string(tok)!r}")

0_pre        ' researchers'
1_pre        ' researcher'
2_pre        ' researcher'
3_pre        'paces'
4_pre        'paces'
5_pre        'hips'
6_pre        ' specializing'
7_pre        ' specializing'
8_pre        ' specializing'
9_pre        ' specializing'
10_pre       ' specializing'
11_pre       ' estimate'
final_post   ' estimate'


Residual stream norm per layer, BOS position separate from the rest

In [23]:
raw, labels = cache.accumulated_resid(return_labels=True)
norms = raw[:, 0].norm(dim=-1)  # [13, 11]
for lab, n in zip(labels, norms):
    print(f"{lab:12s} BOS {n[0]:8.1f}   others (mean) {n[1:].mean():6.1f}")

0_pre        BOS     10.3   others (mean)    5.4
1_pre        BOS    165.2   others (mean)   60.7
2_pre        BOS    641.8   others (mean)   60.7
3_pre        BOS   2577.0   others (mean)   64.9
4_pre        BOS   2775.8   others (mean)   69.5
5_pre        BOS   2929.7   others (mean)   77.1
6_pre        BOS   3026.1   others (mean)   83.0
7_pre        BOS   3084.1   others (mean)   95.9
8_pre        BOS   3119.1   others (mean)  110.8
9_pre        BOS   3141.1   others (mean)  130.5
10_pre       BOS   3152.7   others (mean)  160.6
11_pre       BOS   3155.5   others (mean)  254.3
final_post   BOS    417.4   others (mean)  418.6


Top-5 candidates per layer with probabilities

In [24]:
probs = lens_logits.softmax(-1)
for lab, p in zip(labels, probs):
    top = p.topk(5)
    print(f"{lab:12s}", "  ".join(f"{model.to_string(i)!r}:{v:.2f}" for v, i in zip(top.values, top.indices)))

0_pre        ' researchers':1.00  ' mathemat':0.00  ' scientists':0.00  ' Researchers':0.00  'Researchers':0.00
1_pre        ' researcher':0.20  ' Researchers':0.12  ' researchers':0.11  'Researchers':0.06  'sonian':0.04
2_pre        ' researcher':0.22  'sonian':0.11  ' researchers':0.05  ' Researchers':0.05  ' scientist':0.05
3_pre        'paces':0.17  ' researcher':0.13  ' scientist':0.04  ' Researchers':0.03  'itute':0.03
4_pre        'paces':0.15  'hips':0.08  ' scientist':0.07  ' researcher':0.06  'hip':0.06
5_pre        'hips':0.11  'paces':0.05  ' specializing':0.05  ' compiled':0.04  'hip':0.04
6_pre        ' specializing':0.09  'hips':0.09  ' compiled':0.05  ' uncovered':0.05  ' researchers':0.04
7_pre        ' specializing':0.10  ' Laura':0.08  ' researcher':0.04  ' researchers':0.04  ' worldwide':0.04
8_pre        ' specializing':0.17  ' analyzing':0.07  ' researchers':0.06  ' researcher':0.06  ' authors':0.05
9_pre        ' specializing':0.46  ' analyzing':0.06  ' studying'

Induction heads: a random token sequence repeated twice. In the second half each token is predictable by looking at what followed its previous occurrence.

In [34]:
torch.manual_seed(0)
L = 20
rand = torch.randint(1000, 10000, (1, L))  # mid-frequency token ids, avoids very rare tokens
rep_tokens = torch.cat([torch.tensor([[model.tokenizer.bos_token_id]]), rand, rand], dim=1).to("mps")
rep_logits, rep_cache = model.run_with_cache(rep_tokens)

scores = torch.zeros(model.cfg.n_layers, model.cfg.n_heads)
for l in range(model.cfg.n_layers):
    pat = rep_cache[f"blocks.{l}.attn.hook_pattern"][0]  # [head, dest, src]
    scores[l] = pat.diagonal(offset=-(L - 1), dim1=-2, dim2=-1).mean(-1)
print(scores.round(decimals=2))
top = scores.flatten().topk(5)
print([(i.item() // model.cfg.n_heads, i.item() % model.cfg.n_heads, round(v.item(), 2)) for v, i in zip(top.values, top.indices)])

tensor([[0.0300, 0.0000, 0.0400, 0.0000, 0.0000, 0.0000, 0.0300, 0.0100, 0.0400,
         0.0400, 0.0200, 0.0300],
        [0.0000, 0.0000, 0.0200, 0.0300, 0.0300, 0.0500, 0.0400, 0.0400, 0.0400,
         0.0400, 0.0200, 0.0100],
        [0.0100, 0.0400, 0.0100, 0.0000, 0.0400, 0.0000, 0.0200, 0.0200, 0.0100,
         0.0000, 0.0100, 0.0300],
        [0.0300, 0.0000, 0.0000, 0.0300, 0.0600, 0.0500, 0.0000, 0.0000, 0.0100,
         0.0400, 0.0200, 0.0100],
        [0.0400, 0.0200, 0.0300, 0.0300, 0.0500, 0.0200, 0.0200, 0.0100, 0.0500,
         0.0300, 0.0500, 0.0000],
        [0.5300, 0.8100, 0.0400, 0.0300, 0.0200, 0.8400, 0.0200, 0.0400, 0.2000,
         0.0700, 0.0400, 0.0400],
        [0.0200, 0.0500, 0.0500, 0.0500, 0.0600, 0.0300, 0.0800, 0.0400, 0.0300,
         0.7900, 0.1100, 0.0300],
        [0.0100, 0.2300, 0.7400, 0.0700, 0.0500, 0.0400, 0.1100, 0.1800, 0.0500,
         0.0600, 0.8300, 0.1600],
        [0.0400, 0.4200, 0.0600, 0.1100, 0.0500, 0.0500, 0.2600, 0.0300, 0.1000,

In-context learning made visible: per-token loss should drop sharply in the repeated half

In [29]:
loss = model(rep_tokens, return_type="loss", loss_per_token=True)[0]
print(f"first half {loss[:L].mean():.2f}   second half {loss[L:].mean():.2f}")

first half 12.36   second half 1.10


Look at the strongest induction head: a clean stripe parallel to the diagonal, offset by L - 1

In [32]:
from circuitsvis.attention import attention_pattern

l, h = 5, 5  # replace with the top entry from above
attention_pattern(tokens=model.to_str_tokens(rep_tokens), attention=rep_cache[f"blocks.{l}.attn.hook_pattern"][0, h].cpu())

In [33]:
# First intervention: knock out the five induction heads by zeroing their output (hook_z) and
# measure how much the second-half loss rises. Zero ablation is crude, mean ablation is the
# cleaner standard, but for a first look it shows the mechanism
induction_heads = [(5, 1), (5, 5), (6, 9), (7, 2), (7, 10)]

def make_zero_hook(head):
    def hook(z, hook):  # z: [batch, pos, head, d_head]
        z[:, :, head, :] = 0.0
        return z
    return hook

def second_half_loss(heads):
    fwd_hooks = [(f"blocks.{l}.attn.hook_z", make_zero_hook(h)) for l, h in heads]
    loss = model.run_with_hooks(rep_tokens, return_type="loss", loss_per_token=True, fwd_hooks=fwd_hooks)[0]
    return loss[L:].mean().item()

print(f"clean            {second_half_loss([]):.2f}")
print(f"induction heads  {second_half_loss(induction_heads):.2f}")

# Control: five random non-induction heads from the same layers
torch.manual_seed(1)
pool = [(l, h) for l in (5, 6, 7) for h in range(model.cfg.n_heads) if (l, h) not in induction_heads]
random_heads = [pool[i] for i in torch.randperm(len(pool))[:5]]
print(f"random heads     {second_half_loss(random_heads):.2f}   {random_heads}")

clean            1.10
induction heads  3.86
random heads     1.12   [(6, 0), (7, 11), (6, 4), (7, 7), (7, 9)]
